# Read OHLCV files from S3

Quick notebook to list files under a prefix and load parquet/csv into pandas.

In [1]:
import os
from io import BytesIO
from pathlib import Path

import boto3
import pandas as pd
from dotenv import load_dotenv

load_dotenv(dotenv_path=Path('.env'))

AWS_REGION = os.getenv('MY_AWS_DEFAULT_REGION', 'us-east-2')
BUCKET = os.getenv('MY_AWS_BUCKET_NAME', 'bamboo-analytics-data-platform')
PREFIX = 'bronze/ohlcv/'

s3 = boto3.client(
    's3',
    aws_access_key_id=os.getenv('MY_AWS_ACCESS_KEY_ID'),
    aws_secret_access_key=os.getenv('MY_AWS_SECRET_ACCESS_KEY'),
    region_name=AWS_REGION,
)

print(f'Bucket: {BUCKET}')
print(f'Prefix: {PREFIX}')

Bucket: bamboo-analytics-data-platform
Prefix: bronze/ohlcv/


In [2]:
def list_keys(bucket: str, prefix: str, suffix: str | None = None) -> list[str]:
    keys: list[str] = []
    continuation_token = None

    while True:
        kwargs = {'Bucket': bucket, 'Prefix': prefix}
        if continuation_token:
            kwargs['ContinuationToken'] = continuation_token

        response = s3.list_objects_v2(**kwargs)
        contents = response.get('Contents', [])

        for obj in contents:
            key = obj['Key']
            if suffix is None or key.endswith(suffix):
                keys.append(key)

        if not response.get('IsTruncated'):
            break
        continuation_token = response.get('NextContinuationToken')

    return sorted(keys)


parquet_keys = list_keys(BUCKET, PREFIX, suffix='.parquet')
csv_keys = list_keys(BUCKET, PREFIX, suffix='.csv')

print(f'Found {len(parquet_keys)} parquet files and {len(csv_keys)} csv files')
print('\nLatest parquet files:')
for key in parquet_keys[-5:]:
    print('-', key)

Found 24 parquet files and 0 csv files

Latest parquet files:
- bronze/ohlcv/MSFT/MSFT_20260216_063025.parquet
- bronze/ohlcv/MSFT/MSFT_20260216_063528.parquet
- bronze/ohlcv/MSFT/MSFT_20260216_063546.parquet
- bronze/ohlcv/NVDA/NVDA_20260214_191858.parquet
- bronze/ohlcv/TSLA/TSLA_20260214_191857.parquet


In [10]:
def read_s3_file(bucket: str, key: str) -> pd.DataFrame:
    obj = s3.get_object(Bucket=bucket, Key=key)
    body = obj['Body'].read()

    if key.endswith('.parquet'):
        return pd.read_parquet(BytesIO(body), engine='pyarrow')
    if key.endswith('.csv'):
        return pd.read_csv(BytesIO(body))

    raise ValueError(f'Unsupported file type for key: {key}')


# Pick a file manually or default to latest parquet.
target_key = parquet_keys[-1] if parquet_keys else None

if target_key is None:
    raise ValueError('No parquet files found under prefix')

target_key = "bronze/ohlcv/GOOGL/GOOGL_20260216_063546.parquet"
print('Reading:', target_key)
df = read_s3_file(BUCKET, target_key)
print(df.shape)
df.head()

Reading: bronze/ohlcv/GOOGL/GOOGL_20260216_063546.parquet
(251, 10)


,date,open,high,low,close,volume,dividends,stock_splits,symbol,ingestion_timestamp
0,2024-01-02 00:00:00-05:00,137.511021,138.404265,135.456536,137.133865,23711200,0.0,0.0,GOOGL,2026-02-16 06:35:46.586045
1,2024-01-03 00:00:00-05:00,136.220760,138.582917,136.052037,137.878235,24212100,0.0,0.0,GOOGL,2026-02-16 06:35:46.586045
2,2024-01-04 00:00:00-05:00,137.381978,138.116435,135.327509,135.367203,27137700,0.0,0.0,GOOGL,2026-02-16 06:35:46.586045
3,2024-01-05 00:00:00-05:00,135.724498,136.131427,134.136491,134.712143,22513900,0.0,0.0,GOOGL,2026-02-16 06:35:46.586045
4,2024-01-08 00:00:00-05:00,135.267948,137.967551,135.238174,137.798828,21404000,0.0,0.0,GOOGL,2026-02-16 06:35:46.586045


In [14]:

target_key = "raw/source=yahoo/dataset=ohlcv_1d/year=2024/month=01/ohlcv_1d_2024-01-02.parquet"
print('Reading:', target_key)
df = read_s3_file(BUCKET, target_key)
print(df.shape)
df.head()

Reading: raw/source=yahoo/dataset=ohlcv_1d/year=2024/month=01/ohlcv_1d_2024-01-02.parquet
(3, 12)


,date,open,high,low,close,volume,dividends,stock_splits,symbol,ingestion_timestamp,source,ingestion_ts
0,2024-01-02 00:00:00-05:00,185.225777,186.502522,181.999301,183.731308,82488700,0.0,0.0,AAPL,2026-02-16 18:46:07.512369,yahoo,2026-02-16T18:46:07
1,2024-01-02 00:00:00-05:00,137.511005,138.404250,135.456521,137.133850,23711200,0.0,0.0,GOOGL,2026-02-16 18:46:07.517769,yahoo,2026-02-16T18:46:07
2,2024-01-02 00:00:00-05:00,368.367696,370.377735,361.381857,365.421631,25258600,0.0,0.0,MSFT,2026-02-16 18:46:07.523795,yahoo,2026-02-16T18:46:07
